# Exercise 1.1 - Motion Detection and Car Tracking

This notebook implements an automated traffic monitoring pipeline for **Traffic_Laramie_1.mp4** using OpenCV.

## What this notebook does
- Installs required libraries using notebook package management.
- Applies **frame differencing** and **background subtraction** together.
- Restricts processing to a **Region of Interest (ROI)** on the main street.
- Detects cars with contour filtering to reduce noise/shadows.
- Tracks vehicles across frames with centroid-based ID assignment to avoid duplicate detections.

In [7]:
%pip install -q opencv-python numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Technique summary

### 1) Frame differencing
Frame differencing computes the absolute difference between consecutive grayscale frames:

\begin{equation}
D_t = |I_t - I_{t-1}|
\end{equation}

This highlights moving regions while suppressing static background.

### 2) Background subtraction
A background model (MOG2) estimates the static scene and returns foreground masks for moving objects.

### 3) Combined mask and post-processing
The pipeline combines both motion cues, then applies thresholding and morphology to reduce noise.

### 4) ROI restriction and contour filtering
Only the **main-street ROI** is processed, and small contours are rejected by area thresholding.

### 5) Vehicle tracking
Detected car centroids are associated frame-to-frame using nearest-neighbour matching with a distance gate. This keeps stable IDs and avoids duplicate tracking.

In [8]:
from pathlib import Path
import cv2
import numpy as np

VIDEO_PATH = Path("Traffic_Laramie_1.mp4")
OUTPUT_PATH = Path("EX1_1_tracked_output.mp4")

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH.resolve()}"

cap = cv2.VideoCapture(str(VIDEO_PATH))
ret, sample_frame = cap.read()
cap.release()
assert ret, "Could not read first frame from input video."

height, width = sample_frame.shape[:2]
print(f"Input resolution: {width} x {height}")

Input resolution: 1040 x 600


In [9]:
def main_street_roi_mask(frame_shape):
    h, w = frame_shape[:2]
    pts = np.array([
        [int(0.02 * w), int(0.53 * h)],
        [int(0.78 * w), int(0.53 * h)],
        [int(0.98 * w), int(0.93 * h)],
        [int(0.01 * w), int(0.93 * h)],
    ], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 255)
    return mask, pts


def centroid(box):
    x, y, bw, bh = box
    return (x + bw // 2, y + bh // 2)


def match_tracks(tracks, detections, next_id, max_distance=55, max_missed=12):
    updated_tracks = {}
    used_detection_indices = set()

    track_items = list(tracks.items())
    det_centroids = [centroid(d) for d in detections]

    for track_id, track_data in track_items:
        tx, ty = track_data["centroid"]
        best_idx = None
        best_dist = float("inf")

        for i, (dx, dy) in enumerate(det_centroids):
            if i in used_detection_indices:
                continue
            dist = ((tx - dx) ** 2 + (ty - dy) ** 2) ** 0.5
            if dist < best_dist:
                best_dist = dist
                best_idx = i

        if best_idx is not None and best_dist <= max_distance:
            used_detection_indices.add(best_idx)
            updated_tracks[track_id] = {
                "box": detections[best_idx],
                "centroid": det_centroids[best_idx],
                "missed": 0,
            }
        else:
            missed = track_data["missed"] + 1
            if missed <= max_missed:
                updated_tracks[track_id] = {
                    "box": track_data["box"],
                    "centroid": track_data["centroid"],
                    "missed": missed,
                }

    for i, det in enumerate(detections):
        if i in used_detection_indices:
            continue
        updated_tracks[next_id] = {
            "box": det,
            "centroid": det_centroids[i],
            "missed": 0,
        }
        next_id += 1

    return updated_tracks, next_id


def filter_vehicle_detections(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < 850:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        if w < 24 or h < 20:
            continue
        if h > 2.2 * w:
            continue

        bbox_area = w * h
        if bbox_area < 1400:
            continue

        # Reject partial blobs: full vehicles occupy a larger fraction of their box.
        extent = area / float(bbox_area)
        if extent < 0.35:
            continue

        # Exclude objects clipped by frame borders, which are usually partial vehicles.
        mask_h, mask_w = binary_mask.shape[:2]
        if x <= 0 or y <= 0 or (x + w) >= (mask_w - 1) or (y + h) >= (mask_h - 1):
            continue

        detections.append((x, y, w, h))
    return detections

### Method A: Frame Differencing

This cell performs car detection and tracking using **frame differencing** only (with ROI + contour filtering + centroid tracking).

In [10]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

OUTPUT_DIFF_PATH = Path("EX1_1_frame_diff_tracked.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(OUTPUT_DIFF_PATH), fourcc, fps, (frame_w, frame_h))

roi_mask, roi_polygon = main_street_roi_mask((frame_h, frame_w, 3))

tracks = {}
next_track_id = 1
prev_gray = None
frame_count = 0
window_name = "Frame Differencing Tracking (press q to stop)"
show_enabled = True

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    if prev_gray is None:
        prev_gray = blur
        out.write(frame)
        continue

    diff = cv2.absdiff(blur, prev_gray)
    _, motion_mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
    motion_mask = cv2.bitwise_and(motion_mask, roi_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    detections = filter_vehicle_detections(motion_mask)
    tracks, next_track_id = match_tracks(tracks, detections, next_track_id)

    vis = frame.copy()
    cv2.polylines(vis, [roi_polygon], isClosed=True, color=(0, 0, 255), thickness=2)

    for track_id, track_data in tracks.items():
        x, y, w, h = track_data["box"]
        cx, cy = track_data["centroid"]

        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(vis, (cx, cy), 3, (255, 0, 0), -1)
        cv2.putText(vis, f"ID {track_id}", (x, max(18, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(vis, "Method: Frame Differencing", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(vis, f"Active tracks: {len(tracks)}", (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    out.write(vis)

    if show_enabled:
        try:
            cv2.imshow(window_name, vis)
            if (cv2.waitKey(1) & 0xFF) == ord("q"):
                break
        except cv2.error:
            show_enabled = False
            print("OpenCV GUI display is not available in this environment. Video is still being written.")

    prev_gray = blur

cap.release()
out.release()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass

print(f"Frame differencing complete: {OUTPUT_DIFF_PATH.resolve()}")
print(f"Total created track IDs: {next_track_id - 1}")

Frame differencing complete: C:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise1\EX1_1_frame_diff_tracked.mp4
Total created track IDs: 68


### Method B: Background Subtraction

This cell performs car detection and tracking using **background subtraction (MOG2)** only (with ROI + contour filtering + centroid tracking).

In [11]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

OUTPUT_BGS_PATH = Path("EX1_1_background_subtraction_tracked.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(OUTPUT_BGS_PATH), fourcc, fps, (frame_w, frame_h))

background_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=28, detectShadows=False)
roi_mask, roi_polygon = main_street_roi_mask((frame_h, frame_w, 3))

tracks = {}
next_track_id = 1
frame_count = 0
window_name = "Background Subtraction Tracking (press q to stop)"
show_enabled = True

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    fg_mask = background_subtractor.apply(blur)
    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)
    fg_mask = cv2.bitwise_and(fg_mask, roi_mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    detections = filter_vehicle_detections(fg_mask)
    tracks, next_track_id = match_tracks(tracks, detections, next_track_id)

    vis = frame.copy()
    cv2.polylines(vis, [roi_polygon], isClosed=True, color=(0, 0, 255), thickness=2)

    for track_id, track_data in tracks.items():
        x, y, w, h = track_data["box"]
        cx, cy = track_data["centroid"]

        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(vis, (cx, cy), 3, (255, 0, 0), -1)
        cv2.putText(vis, f"ID {track_id}", (x, max(18, y - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(vis, "Method: Background Subtraction", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(vis, f"Active tracks: {len(tracks)}", (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    out.write(vis)

    if show_enabled:
        try:
            cv2.imshow(window_name, vis)
            if (cv2.waitKey(1) & 0xFF) == ord("q"):
                break
        except cv2.error:
            show_enabled = False
            print("OpenCV GUI display is not available in this environment. Video is still being written.")

cap.release()
out.release()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass

print(f"Background subtraction complete: {OUTPUT_BGS_PATH.resolve()}")
print(f"Total created track IDs: {next_track_id - 1}")

Background subtraction complete: C:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise1\EX1_1_background_subtraction_tracked.mp4
Total created track IDs: 24


In [12]:
for method_name, output_path in [
    ("Frame Differencing", Path("EX1_1_frame_diff_tracked.mp4")),
    ("Background Subtraction", Path("EX1_1_background_subtraction_tracked.mp4")),
]:
    preview_cap = cv2.VideoCapture(str(output_path))
    frame_total = int(preview_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_out = preview_cap.get(cv2.CAP_PROP_FPS)
    ret, preview_frame = preview_cap.read()
    preview_cap.release()

    print(f"\n{method_name}")
    if ret:
        print(f"Output video created: {output_path.resolve()}")
        print(f"Output FPS: {fps_out:.2f}")
        print(f"Output frame count: {frame_total}")
        print(f"Preview frame shape: {preview_frame.shape}")
    else:
        print("Could not read output video for preview.")


Frame Differencing
Output video created: C:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise1\EX1_1_frame_diff_tracked.mp4
Output FPS: 25.00
Output frame count: 3280
Preview frame shape: (600, 1040, 3)

Background Subtraction
Output video created: C:\Users\Enzo\Desktop\SIM-UOL\Y3S1\CM3065 - Intelligent Signal Processing\EndTerm\Exercise1\EX1_1_background_subtraction_tracked.mp4
Output FPS: 25.00
Output frame count: 1318
Preview frame shape: (600, 1040, 3)


## Brief analysis

- The combined use of frame differencing and MOG2 reduces missed detections compared with using only one method.
- Gaussian blur before differencing reduces high-frequency noise.
- ROI restriction avoids false positives from buildings and sky regions.
- Area and shape filtering suppresses small artifacts and non-vehicle blobs.
- Centroid-based track association limits duplicate tracking IDs.

## Further development

- Add a counting line for downtown-to-city-centre direction counting (Exercise 1.2).
- Replace nearest-neighbour tracking with Kalman + Hungarian assignment for denser traffic.
- Improve robustness under illumination changes with adaptive thresholds.
- Log per-track trajectories and speed estimates for richer traffic analytics.